In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [ ]:
# df = pd.read_csv("../data/sintetis/dataset_irigasi.csv")
df = pd.read_csv("../data/dataset_irigasi.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 1768 baris, 5 kolom


,soil_moisture,soil_temperature,air_temperature,air_humidity,irrigation_action
0,82.5,25.3,24.8,58.0,0
1,80.5,25.2,24.8,57.1,0
2,77.9,25.6,25.3,53.5,0
3,78.2,25.8,25.0,55.4,0
4,75.1,26.0,26.0,51.5,0


# Pisahkan Fitur & Label

In [15]:
FEATURES = ["soil_moisture", "soil_temperature", "air_temperature", "air_humidity"]

X = df[FEATURES].values
y = df["irrigation_action"].values

print(f"Fitur: {FEATURES}")
print(f"Label 0 (tidak siram): {(y==0).sum()}")
print(f"Label 1 (siram):       {(y==1).sum()}")
print(f"Rasio siram: {y.mean()*100:.1f}%")

Fitur: ['soil_moisture', 'soil_temperature', 'air_temperature', 'air_humidity']
Label 0 (tidak siram): 1060
Label 1 (siram):       708
Rasio siram: 40.0%


In [16]:
if len(set(y)) < 2:
    print("⚠️ BAHAYA: cuma 1 kelas! Data belum variatif, model gak bisa dilatih.")
else:
    print(f"OK — {len(set(y))} kelas. Lanjut.")

OK — 2 kelas. Lanjut.


# Split Train/Test

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 1414, Test: 354


# Training Random Forest

In [18]:
model = RandomForestClassifier(
    n_estimators=15,     
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


# Cross Validation

In [19]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
print(f"CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1: 0.6577 (+/- 0.0147)


# Evaluasi di Test Set

In [20]:
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Tidak siram', 'Siram'])}")

Accuracy: 0.7288
F1: 0.6923

Confusion Matrix:
[[150  62]
 [ 34 108]]

              precision    recall  f1-score   support

 Tidak siram       0.82      0.71      0.76       212
       Siram       0.64      0.76      0.69       142

    accuracy                           0.73       354
   macro avg       0.73      0.73      0.72       354
weighted avg       0.74      0.73      0.73       354



# Feature importance

In [21]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:18s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  soil_moisture      0.6819  ██████████████████████████████████
  air_temperature    0.1109  █████
  soil_temperature   0.1084  █████
  air_humidity       0.0988  ████


# Test Prediksi Manual

In [22]:
# Skenario 1: tanah normal (62%) tapi panas & kering → harusnya SIRAM
s1 = [[62, 35, 38, 40]]
# Skenario 2: tanah agak kering (58%) tapi lembab & sejuk → harusnya TUNDA
s2 = [[58, 24, 25, 90]]

for i, s in enumerate([s1, s2], 1):
    pred = model.predict(s)[0]
    conf = model.predict_proba(s)[0].max()
    print(f"Skenario {i}: {s[0]} → {'SIRAM' if pred else 'TIDAK'} ({conf:.1%})")

Skenario 1: [62, 35, 38, 40] → SIRAM (69.3%)
Skenario 2: [58, 24, 25, 90] → SIRAM (60.8%)


# Simpan Model

In [23]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/rf_irigasi.joblib")
print("Model tersimpan: models/rf_irigasi.joblib")

Model tersimpan: models/rf_irigasi.joblib


# Convert ke esp

In [24]:
from micromlgen import port

c_code = port(model)
with open("model_irigasi.h", "w") as f:
    f.write(c_code)
print("Tersimpan: model_irigasi.h (siap #include di Arduino IDE)")
print(f"Ukuran: {len(c_code)} karakter")

Tersimpan: model_irigasi.h (siap #include di Arduino IDE)
Ukuran: 236069 karakter
